# 8) Frequency and reserves — Exercises

**Goals:** write the closed-form results, check the FCR timing rule, weigh inertia against fast response, tune frequency restoration, and see what a small island faces that a large interconnected system does not.

The setup cell holds the lecture's `simulate(...)` — now with an optional **fast reserve** `fast_gw` (for example batteries, time constant `T_fast_s` = 0.5 s, fully activated at 200 mHz like FCR) — plus `nadir(run)` and a small `bisect(f, low, high, target, tol)` helper. Frequencies in Hz, power in GW, the reference case S = 300 GW, H = 5 s, FCR 3 GW, loss 3 GW.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [ ]:
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp

F0 = 50.0


def simulate(loss_gw, S_gw=300.0, H_s=5.0, fcr_gw=3.0, fcr_full_hz=0.2, T_fcr_s=6.0, self_regulation=0.01,
             Ki_afrr=0.0, fast_gw=0.0, T_fast_s=0.5, T_end=120.0, step=0.01):
    """The lecture's one-area frequency model. fast_gw: a fast reserve (e.g. batteries) with time constant T_fast_s,
    fully activated at the same 200 mHz. Returns a DataFrame indexed by time (s)."""
    K_fcr = fcr_gw / fcr_full_hz
    D = self_regulation * S_gw
    K_fast = fast_gw / fcr_full_hz if fast_gw else 0.0

    def rhs(t, x):
        df, p_fcr, p_afrr, p_fast = x
        target_fcr = np.clip(-K_fcr * df, -fcr_gw, fcr_gw)
        target_fast = np.clip(-K_fast * df, -fast_gw, fast_gw)
        response = p_fcr + p_afrr + p_fast - D * df
        return [F0 / (2 * H_s * S_gw) * (response - loss_gw), (target_fcr - p_fcr) / T_fcr_s, -Ki_afrr * df,
                (target_fast - p_fast) / T_fast_s]

    t = np.linspace(0, T_end, int(round(T_end / step)) + 1)
    sol = solve_ivp(rhs, (0, T_end), [0.0, 0.0, 0.0, 0.0], t_eval=t, max_step=step, rtol=1e-9, atol=1e-11)
    df, p_fcr, p_afrr, p_fast = sol.y
    return pd.DataFrame({"df_Hz": df, "FCR_GW": p_fcr, "aFRR_GW": p_afrr, "fast_GW": p_fast,
                         "load_response_GW": -D * df}, index=pd.Index(sol.t, name="t_s"))


def nadir(run):
    return float(run.df_Hz.min())


def bisect(f, low, high, target, tol):
    """Smallest x in [low, high] with f(x) >= target, for f increasing in x."""
    while high - low > tol:
        mid = (low + high) / 2
        low, high = (mid, high) if f(mid) < target else (low, mid)
    return high

### Warm-ups

**1. The two closed-form results**

Write `initial_rocof(loss_gw, S_gw, H_s)` and `settled_deviation(loss_gw, fcr_gw, S_gw, self_regulation)` (FCR fully activated at 200 mHz, not saturated). For the reference incident, compare the settled deviation with and without load self-regulation, and check both functions against `simulate`.

In [ ]:
def initial_rocof(loss_gw: float, S_gw: float, H_s: float) -> float:
    ...  # YOUR CODE HERE


def settled_deviation(loss_gw: float, fcr_gw: float, S_gw: float, self_regulation: float = 0.01) -> float:
    ...  # YOUR CODE HERE

In [ ]:
run = simulate(3.0)
assert np.isclose(initial_rocof(3.0, 300.0, 5.0), (run.df_Hz.iloc[1] - run.df_Hz.iloc[0]) / 0.01, atol=1e-3)
assert np.isclose(settled_deviation(3.0, 3.0, 300.0), run.df_Hz.iloc[-1], atol=1e-4)
print(f"settled: {settled_deviation(3.0, 3.0, 300.0) * 1000:.1f} mHz with load self-regulation, "
      f"{settled_deviation(3.0, 3.0, 300.0, 0.0) * 1000:.1f} mHz without")

**2. The FCR timing rule**

Continental Europe requires FCR to deliver at least 50 % of a full activation within 15 s and 100 % within 30 s. With the model's first-order lag, the share delivered after a step is 1 − exp(−t / T). Write `fcr_shares(T)` returning the shares at 15 s and 30 s, tabulate them for T in [2, 6, 10, 20] s, and compute the largest T that delivers 50 % at 15 s and the largest that delivers 99 % at 30 s.

In [ ]:
def fcr_shares(T_s: float) -> tuple[float, float]:
    ...  # YOUR CODE HERE

In [ ]:
table = pd.DataFrame({T: dict(zip(["share_at_15s", "share_at_30s"], fcr_shares(T))) for T in [2, 6, 10, 20]}).T
largest_50 = 15 / np.log(2)
largest_99 = 30 / np.log(100)
assert np.isclose(fcr_shares(largest_50)[0], 0.5) and np.isclose(fcr_shares(largest_99)[1], 0.99)
print(f"largest T for 50 % at 15 s: {largest_50:.1f} s; for 99 % at 30 s: {largest_99:.2f} s")
table.rename_axis("T (s)").round(3)

### Core

**3. Inertia or speed?**

Suppose inertia falls from H = 5 s to H = 2 s. The reference incident's nadir deepens from −0.247 to −0.363 Hz. Find, by bisection to 0.01 GW, how much **fast reserve** (T = 0.5 s) restores the H = 5 s nadir at H = 2 s. Compare: how much **synchronous capacity** would have to be added to the 300 GW system to raise its inertia back to H = 5 s, if the added machines have H = 5 s themselves? Write `inertia_or_speed()`.

In [ ]:
def inertia_or_speed() -> dict:
    """Fast reserve needed at H = 2 s to match the H = 5 s nadir, and the equivalent synchronous capacity."""
    ...  # YOUR CODE HERE

In [ ]:
speed = inertia_or_speed()
assert speed["fast_reserve_GW"] < 3.0 and speed["synchronous_capacity_GW"] > 100
pd.Series(speed).round(3)

**4. Restoring 50 Hz**

For aFRR gains `Ki_afrr` in [0.02, 0.05, 0.10, 0.15, 0.30] GW/(Hz·s), simulate 15 minutes after the reference incident (use `step=0.05`) and report the deviation after 5 and after 15 minutes, the largest overshoot above 50 Hz, and the FCR still active after 15 minutes. Which gains meet the 15-minute restoration requirement (say, within 1 mHz)? Write `afrr_gains(gains)`.

In [ ]:
def afrr_gains(gains=(0.02, 0.05, 0.10, 0.15, 0.30)) -> pd.DataFrame:
    ...  # YOUR CODE HERE

In [ ]:
gains = afrr_gains()
assert abs(gains.loc[0.15, "deviation_15min_mHz"]) < 1 and abs(gains.loc[0.02, "deviation_15min_mHz"]) > 1
gains.round(2)

**5. Interpretation**

How much fast reserve replaces the lost inertia, compared with the synchronous capacity that would do the same — and what does that comparison miss? What limits the aFRR gain from above, if a faster restoration is always better for the frequency?

*Your answer here.*

### Challenge

**6. Croatia as an island**

Imagine a 3 GW system (roughly Croatia's load) running as an island, with H = 5 s and 1 % per Hz load self-regulation, losing a 500 MW unit. (a) Give it 600 MW of FCR with T = 6 s: what are the initial RoCoF and the nadir? (b) How much FCR with the same 6 s lag would keep the nadir above −0.8 Hz? Find it by bisection to 0.01 GW, and report how much of that reserve is actually delivered at the nadir. (c) How much **fast** reserve (T = 0.5 s), added to the 600 MW of FCR, keeps the nadir above −0.8 Hz? (d) For contrast: the same 500 MW loss in the 300 GW interconnected system of the lecture. Write `island()`.

The model is linear and has no protection: in a real system, frequencies far outside 49–51 Hz trigger automatic load shedding and generator tripping long before the model's numbers are reached. Read large deviations as "the system would not hold", not as a prediction.

In [ ]:
def island() -> dict:
    """The 3 GW island losing 500 MW, with different reserves, and the interconnected comparison."""
    ...  # YOUR CODE HERE

In [ ]:
result = island()
assert result["slow_FCR_needed_GW"] > 3 * result["fast_reserve_needed_GW"], "slow reserve needs far more capacity"
assert result["nadir_interconnected_Hz"] > -0.1
pd.Series(result).round(3)

**7. Interpretation**

Why does slow FCR need so much more capacity than fast reserve to hold the island, although it delivers only part of it at the nadir? What does the comparison with the interconnected case say about why Croatia — and every small system — is part of a large synchronous area?

*Your answer here.*